# Custom Preprocessing
In this example, we demonstrate how to use custom preprocessing steps in the model object. We show how to split the input features and define a custom preprocessing /feature selection routine.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn import pipeline as sklearn_pipeline
from sklearn.model_selection import GroupKFold

import mother.ml as ml
from mother.cv import core
from mother import feature_generation as fg

from mother.preprocessing import SmilesToMolTransformer, StandardizerTransformer

## Usual regression workflow
.. as pointed out in the regression example. However, we add a string-column to the dataset. This column could be used as a categoric feature in the model, however, for demonstration pruposes we show in this example how to handle this column in a custom preprocessing pipeline in the "model" object.


In [2]:
input_file: Path = Path("../freesolv_train.csv")
data: pd.DataFrame = pd.read_csv(input_file, sep=",")
data = data.head(100)  # limit data to 100 rows for testing
data = data[["iupac", "smiles", "expt"]]

In [3]:
# add a string column to the data
rng = np.random.default_rng(42)
data["string_col"] = rng.choice(["A", "B", "C"], data.shape[0])

In [4]:
data

,iupac,smiles,expt,string_col
0,"4-methoxy-N,N-dimethyl-benzamide",CN(C)C(=O)c1ccc(cc1)OC,-11.01,A
1,methanesulfonyl chloride,CS(=O)(=O)Cl,-4.87,C
2,3-methylbut-1-ene,CC(C)C=C,1.83,B
3,2-ethylpyrazine,CCc1cnccn1,-5.45,B
4,heptan-1-ol,CCCCCCCO,-4.21,B
5,"3,5-dimethylphenol",Cc1cc(cc(c1)O)C,-6.27,C
6,"2,3-dimethylbutane",CC(C)C(C)C,2.34,A
7,2-methylpentan-2-ol,CCCC(C)(C)O,-3.92,C
8,"1,2-dimethylcyclohexane",C[C@@H]1CCCC[C@@H]1C,1.58,A
9,butan-2-ol,CC[C@H](C)O,-4.62,A


In [5]:
preprocessor: sklearn_pipeline.Pipeline = sklearn_pipeline.Pipeline(
    [
        (
            "smiles_standardizer",
            StandardizerTransformer(flags=["STANDARDIZE", "DESALT", "NEUTRALIZE"]),
        ),
        ("smiles_to_mol", SmilesToMolTransformer()),
        # Add other column transformations here if needed
    ],
    memory=None,
).set_output(transform="pandas")

structure_data: pd.Series = data["smiles"]
mol_data: pd.DataFrame = preprocessor.fit_transform(structure_data)

[08:10:59] Initializing Normalizer
[08:10:59] Initializing Normalizer
[08:10:59] Initializing Normalizer
[08:10:59] Initializing MetalDisconnector
[08:10:59] Initializing Normalizer
[08:10:59] Initializing MetalDisconnector
[08:10:59] Initializing Normalizer


In [6]:
feature_generator = sklearn_pipeline.FeatureUnion(
    transformer_list=[
        ("maccs", fg.MaccsFingerprints()),
        ("morgan", fg.MorganFingerprints()),
        ("desc", fg.ChemicalDescriptors()),
    ]
).set_output(transform="pandas")


features: pd.DataFrame = feature_generator.fit_transform(mol_data["Molecule"])

Use one of the available strategies to define groups for cross-validation.

In [7]:
# cv grouping
groups_engine = core.HdbscanGroupingFromMols(scaffold="Murcko", radius=2, fp_size=2048, include_chirality=False)

groups: pd.DataFrame = groups_engine.set_output(transform="pandas").fit_transform(mol_data["Molecule"])
groups.head()

cv = GroupKFold(n_splits=5)

print(f"{groups['hdbscan-group'].nunique()} groups found")

3 groups found


## Model Training
The model consists of two steps:
1a. Feature selection using the __FeatureSelector__ for columns from the feature generator
1b. Custom preprocessing for the string column.
2. Model training using the __CatboostRegressorMother__

In [8]:
# defining a dummy custom preprocessing step for the string column. Replace this with your own preprocessing steps, if needed.

from sklearn.base import BaseEstimator, TransformerMixin


class CustomStringPreprocessor(BaseEstimator, TransformerMixin):
    """For demonstration: this is an implementation of a one-hot-encoder for a string column."""

    def __init__(self):
        """
        Initialize the CustomStringPreprocessor.

        This method is intentionally empty as no parameters or state need to be
        initialized. The transformer operates on fixed categories ['A', 'B', 'C']
        and doesn't require configuration.
        """
        pass

    def fit(self, X, y=None):  # noqa: ARG002 - y parameter required for sklearn compatibility
        self.is_fitted_ = True  # tell sklearn that the transformer is fitted
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        Y = np.zeros((X.shape[0], 3))
        for i, x in enumerate(X):
            if x == "A":
                Y[i, 0] = 1
            elif x == "B":
                Y[i, 1] = 1
            elif x == "C":
                Y[i, 2] = 1
        return Y

    def get_feature_names_out(self, *args, **kwargs) -> list:  # noqa: ARG002 - parameters required for compatibility
        return [f"OHE_{i}" for i in ("A", "B", "C")]

In [9]:
# test the object
prep: TransformerMixin = CustomStringPreprocessor().set_output(transform="pandas")
prep.fit_transform(data["string_col"])

,OHE_A,OHE_B,OHE_C
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,0.0,1.0,0.0
3,0.0,1.0,0.0
4,0.0,1.0,0.0
5,0.0,0.0,1.0
6,1.0,0.0,0.0
7,0.0,0.0,1.0
8,1.0,0.0,0.0
9,1.0,0.0,0.0


In [10]:
# Use the skl ColumnTransformer to combine the custom preprocessing with the usual feature selection
from mother import pipeline_utils

model_settings = {
    "feature_selection_flags": [
        "DROP_CORRELATED",
        "DROP_CONSTANT",
        "DROP_DUPLICATES",
        "DROP_UNIMPORTANT",
    ],
    "feature_selection_threshold": 0,
    "correlation_threshold": 0.9,
    "algorithm": "catboost",
    "feature_selection_type": "catboost",
    "model_type": "regression",
    "target_type": "single_target",
}
pipeline_settings = {
    "remainder": "passthrough",
    "verbose_feature_names_out": False,
}

custom_model_step = ml.ColumnTransformerWithHyperparameterRooting(
    transformers=[
        ("custom_string_preprocessor", CustomStringPreprocessor(), ["string_col"]),
        (
            "feature_selector",
            pipeline_utils.get_feature_selection_pipeline(
                settings=model_settings, pipeline_settings=pipeline_settings, cv=cv
            ),
            list(features),
        ),
    ],
    **pipeline_settings,
).set_output(transform="pandas")

In [11]:
model = ml.PipelineWithHyperparameterRooting(
    [
        ("custom_model_step", custom_model_step),
        (
            "ml_model",
            ml.CatboostRegressorMother(target_type="single_target", logging_level="Silent"),
        ),
    ]
)
# fit the model to data

x = pd.concat((features, data["string_col"]), axis=1)

model.fit(x, data["expt"])

targets_pred = model.predict(x)

In [12]:
model["custom_model_step"]["custom_string_preprocessor"]

CustomStringPreprocessor()